# Tarea Nº 1 — Inteligencia Artificial
## Ítem 2: HMM

**Estudiante:** Maximiliano Solorza  
**Fecha de entrega:** Domingo 19 de Abril de 2026

In [1]:
!pip install hmmlearn numpy pandas

### A. Definición formal de estados y observaciones

**Estados ocultos (6):**
$$S = \{\text{WALKING},\ \text{WALKING\_UPSTAIRS},\ \text{WALKING\_DOWNSTAIRS},\ \text{SITTING},\ \text{STANDING},\ \text{LAYING}\}$$

**Índices utilizados (según enunciado):** se calculan a partir de la magnitud 
del acelerómetro (`tBodyAccMag`) y la magnitud del giroscopio (`tBodyGyroMag`):

$$I_{acc} = \frac{\texttt{tBodyAccMag-mean()} + \texttt{tBodyAccMag-std()}}{2}$$

$$I_{gyro} = \frac{\texttt{tBodyGyroMag-mean()} + \texttt{tBodyGyroMag-std()}}{2}$$

Cada índice se discretiza en 3 niveles (**bajo**, **medio**, **alto**) usando 
**terciles empíricos** (cuantiles 1/3 y 2/3). Este criterio produce clases 
*equiprobables a priori*, evita sesgos por colas pesadas y no asume normalidad. 
El alfabeto de observaciones es:

$$V = \{(l_1, l_2) : l_1, l_2 \in \{\text{bajo},\text{medio},\text{alto}\}\},\quad |V| = 9$$

Mapeo a enteros: $o = 3 \cdot I_{acc,d} + I_{gyro,d} \in \{0,\dots,8\}$.

### Imports

In [1]:
import numpy as np
import pandas as pd
from hmmlearn.hmm import CategoricalHMM

np.random.seed(42)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

### Carga de datos

In [2]:
BASE = "UCI HAR Dataset"


features = pd.read_csv(f"{BASE}/features.txt", sep=r"\s+", header=None, names=["idx", "name"])
features["name"] = pd.Series(features["name"]).astype(str)
features["name"] = features["name"] + "_" + features.groupby("name").cumcount().astype(str)
features.loc[features["name"].str.endswith("_0"), "name"] = features["name"].str[:-2]

activity_labels = pd.read_csv(f"{BASE}/activity_labels.txt", sep=r"\s+",
                               header=None, names=["id", "activity"])

COLS_NEEDED = ["tBodyAccMag-mean()", "tBodyAccMag-std()",
               "tBodyGyroMag-mean()", "tBodyGyroMag-std()"]

def load_split(split):
    X = pd.read_csv(f"{BASE}/{split}/X_{split}.txt", sep=r"\s+",
                    header=None, names=features["name"].values)
    y = pd.read_csv(f"{BASE}/{split}/y_{split}.txt", header=None, names=["label"])
    s = pd.read_csv(f"{BASE}/{split}/subject_{split}.txt", header=None, names=["subject"])
    return pd.concat([s, y, X[COLS_NEEDED]], axis=1)

df = pd.concat([load_split("train"), load_split("test")], ignore_index=True)

id2act = dict(zip(activity_labels["id"], activity_labels["activity"]))
df["activity"] = df["label"].map(id2act)

print(f"Filas: {len(df)} | Sujetos: {df['subject'].nunique()} | Actividades: {df['label'].nunique()}")
df.head()

Filas: 10299 | Sujetos: 30 | Actividades: 6


,subject,label,tBodyAccMag-mean(),tBodyAccMag-std(),tBodyGyroMag-mean(),tBodyGyroMag-std(),activity
0,1,5,-0.9594,-0.9506,-0.9690,-0.9643,STANDING
1,1,5,-0.9793,-0.9761,-0.9807,-0.9838,STANDING
2,1,5,-0.9837,-0.9880,-0.9763,-0.9861,STANDING
3,1,5,-0.9865,-0.9864,-0.9821,-0.9874,STANDING
4,1,5,-0.9928,-0.9913,-0.9852,-0.9891,STANDING


### B. Discretización por terciles (9 observaciones)

In [3]:

df["I_acc"]  = (df["tBodyAccMag-mean()"] + df["tBodyAccMag-std()"])  / 2.0
df["I_gyro"] = (df["tBodyGyroMag-mean()"] + df["tBodyGyroMag-std()"]) / 2.0

# Discretización en 3 niveles por terciles (bajo=0, medio=1, alto=2)
def terciles(s):
    q1, q2 = s.quantile(1/3), s.quantile(2/3)
    return pd.cut(s, bins=[-np.inf, q1, q2, np.inf], labels=[0, 1, 2]).astype(int)

df["I_acc_d"]  = terciles(df["I_acc"])
df["I_gyro_d"] = terciles(df["I_gyro"])

# Observación: combinación de ambos índices discretizados → 9 valores posibles (0..8)
df["obs"] = (3 * df["I_acc_d"] + df["I_gyro_d"]).astype(int)

LVL = {0: "bajo", 1: "medio", 2: "alto"}
OBS_LABELS = [f"Iacc={LVL[i]},Igyro={LVL[j]}" for i in range(3) for j in range(3)]

print("Obs únicas:", sorted(df["obs"].unique()))
print("Etiquetas de observaciones:", OBS_LABELS)

Obs únicas: [np.int64(0), np.int64(1), np.int64(3), np.int64(4), np.int64(5), np.int64(7), np.int64(8)]
Etiquetas de observaciones: ['Iacc=bajo,Igyro=bajo', 'Iacc=bajo,Igyro=medio', 'Iacc=bajo,Igyro=alto', 'Iacc=medio,Igyro=bajo', 'Iacc=medio,Igyro=medio', 'Iacc=medio,Igyro=alto', 'Iacc=alto,Igyro=bajo', 'Iacc=alto,Igyro=medio', 'Iacc=alto,Igyro=alto']


### B. Construcción de secuencias (una por sujeto)

In [5]:
STATES = ["WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
          "SITTING", "STANDING", "LAYING"]
state2idx = {s: i for i, s in enumerate(STATES)}
N_STATES, N_OBS = 6, 9

sequences = []
for subj in sorted(df["subject"].unique()):
    sub = df[df["subject"] == subj].sort_index().reset_index(drop=True)
    sequences.append((sub["label"].values - 1, sub["obs"].values.astype(int)))

print(f"Secuencias: {len(sequences)} | longitud media: {np.mean([len(s[0]) for s in sequences]):.1f}")

Secuencias: 30 | longitud media: 343.3


### B. Estimación manual de $\pi$, $A$ y $B$ por conteo y frecuencias relativas

$$\pi_i = \frac{\#\{\text{secuencias que empiezan en }i\}}{\#\text{secuencias}},\quad
A_{ij} = \frac{\#(i\to j)}{\sum_k \#(i\to k)},\quad
B_{ik} = \frac{\#(i, o_k)}{\sum_m \#(i, o_m)}$$

Se añade un suavizado Laplace $\varepsilon = 10^{-6}$ para evitar probabilidades nulas.

In [6]:
pi = np.zeros(N_STATES)
A  = np.zeros((N_STATES, N_STATES))
B  = np.zeros((N_STATES, N_OBS))

for state_seq, obs_seq in sequences:
    pi[state_seq[0]] += 1
    for t in range(len(state_seq) - 1):
        A[state_seq[t], state_seq[t+1]] += 1
    for t in range(len(state_seq)):
        B[state_seq[t], obs_seq[t]] += 1

eps = 1e-6
pi = (pi + eps) / (pi.sum() + N_STATES * eps)
A  = (A + eps)  / (A.sum(axis=1, keepdims=True) + N_STATES * eps)
B  = (B + eps)  / (B.sum(axis=1, keepdims=True) + N_OBS * eps)

print("Vector inicial π:")
print(pd.Series(pi, index=STATES).round(4))

Vector inicial π:
WALKING              0.0000
WALKING_UPSTAIRS     0.0000
WALKING_DOWNSTAIRS   0.0000
SITTING              0.0000
STANDING             1.0000
LAYING               0.0000
dtype: float64


In [7]:
print("Matriz de transición A:")
pd.DataFrame(A, index=STATES, columns=STATES).round(4)

Matriz de transición A:


,WALKING,WALKING_UPSTAIRS,WALKING_DOWNSTAIRS,SITTING,STANDING,LAYING
WALKING,0.9652,0.0000,0.0348,0.0000,0.0000,0.0000
WALKING_UPSTAIRS,0.0000,0.9664,0.0138,0.0000,0.0198,0.0000
WALKING_DOWNSTAIRS,0.0000,0.0549,0.9451,0.0000,0.0000,0.0000
SITTING,0.0000,0.0000,0.0000,0.9657,0.0000,0.0343
STANDING,0.0000,0.0000,0.0000,0.0315,0.9685,0.0000
LAYING,0.0309,0.0000,0.0000,0.0005,0.0000,0.9686


In [8]:
print("Matriz de emisión B:")
pd.DataFrame(B, index=STATES, columns=OBS_LABELS).round(4)

Matriz de emisión B:


,"Iacc=bajo,Igyro=bajo","Iacc=bajo,Igyro=medio","Iacc=bajo,Igyro=alto","Iacc=medio,Igyro=bajo","Iacc=medio,Igyro=medio","Iacc=medio,Igyro=alto","Iacc=alto,Igyro=bajo","Iacc=alto,Igyro=medio","Iacc=alto,Igyro=alto"
WALKING,0.0000,0.0000,0.0000,0.0000,0.3664,0.1150,0.0000,0.0470,0.4715
WALKING_UPSTAIRS,0.0000,0.0000,0.0000,0.0000,0.1224,0.1367,0.0000,0.1120,0.6289
WALKING_DOWNSTAIRS,0.0000,0.0000,0.0000,0.0000,0.0206,0.0199,0.0000,0.1024,0.8570
SITTING,0.5909,0.0568,0.0000,0.0597,0.2864,0.0000,0.0000,0.0056,0.0006
STANDING,0.5021,0.0714,0.0000,0.0367,0.3888,0.0005,0.0000,0.0005,0.0000
LAYING,0.5571,0.0545,0.0000,0.0859,0.2845,0.0000,0.0000,0.0149,0.0031


### B. Modelo `hmmlearn` con parámetros cargados manualmente

In [9]:
model = CategoricalHMM(n_components=N_STATES, n_features=N_OBS, init_params="")
model.startprob_    = pi
model.transmat_     = A
model.emissionprob_ = B
print("Modelo CategoricalHMM construido con π, A y B estimados manualmente.")

Modelo CategoricalHMM construido con π, A y B estimados manualmente.


### C. Forward–Backward sobre subsecuencia real (T = 15 ≥ 10)

In [10]:
T = 15
state_true, obs_true = sequences[0]  
obs_sub   = obs_true[:T].reshape(-1, 1)
state_sub = state_true[:T]

print("Observaciones :", obs_true[:T].tolist())
print("Estados reales:", [STATES[s] for s in state_sub])

Observaciones : [3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4]
Estados reales: ['STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING', 'STANDING']


In [12]:
logprob, posteriors = model.score_samples(obs_sub)
print(f"\nlog P(O | λ) = {logprob:.4f}")
print(f"P(O | λ)     = {np.exp(logprob):.4e}\n")


log P(O | λ) = -12.7560
P(O | λ)     = 2.8850e-06



**4 consultas** sobre $P(q_t = s \mid O_{1:T})$:

In [15]:
queries = [
    (0,  "WALKING"),
    (3,  "SITTING"),
    (7,  "STANDING"),
    (12, "LAYING"),
]

print(f"{'t':>4} | {'Estado consultado':<22} | {'P(q_t | O)':>12} | {'Estado real'}")
print("-" * 65)

for t, s in queries:
    p = posteriors[t, state2idx[s]]
    real = STATES[state_sub[t]]
    print(f"  {t:>2} | {s:<22} | {p:>12.6f} | {real}")

   t | Estado consultado      |   P(q_t | O) | Estado real
-----------------------------------------------------------------
   0 | WALKING                |     0.000000 | STANDING
   3 | SITTING                |     0.263561 | STANDING
   7 | STANDING               |     0.537680 | STANDING
  12 | LAYING                 |     0.088409 | STANDING


### D. Viterbi: camino más probable y probabilidad de la secuencia

In [18]:
logprob_path, path = model.decode(obs_sub, algorithm="viterbi")
logprob_obs        = model.score(obs_sub)

print("Secuencia más probable (Viterbi):")
for t, s in enumerate(path):
    print(f"  t={t:>2}: {STATES[s]:<25} | obs={OBS_LABELS[obs_true[t]]}")


print(f"\nlog P(q*, O | λ) = {logprob_path:.4f}  →  P = {np.exp(logprob_path):.4e}")
print(f"log P(O | λ)     = {logprob_obs:.4f}   →  P = {np.exp(logprob_obs):.4e}")

Secuencia más probable (Viterbi):
  t= 0: STANDING                  | obs=Iacc=medio,Igyro=bajo
  t= 1: STANDING                  | obs=Iacc=bajo,Igyro=bajo
  t= 2: STANDING                  | obs=Iacc=bajo,Igyro=bajo
  t= 3: STANDING                  | obs=Iacc=bajo,Igyro=bajo
  t= 4: STANDING                  | obs=Iacc=bajo,Igyro=bajo
  t= 5: STANDING                  | obs=Iacc=bajo,Igyro=bajo
  t= 6: STANDING                  | obs=Iacc=bajo,Igyro=bajo
  t= 7: STANDING                  | obs=Iacc=bajo,Igyro=bajo
  t= 8: STANDING                  | obs=Iacc=bajo,Igyro=bajo
  t= 9: STANDING                  | obs=Iacc=bajo,Igyro=bajo
  t=10: STANDING                  | obs=Iacc=bajo,Igyro=bajo
  t=11: STANDING                  | obs=Iacc=bajo,Igyro=bajo
  t=12: STANDING                  | obs=Iacc=bajo,Igyro=bajo
  t=13: STANDING                  | obs=Iacc=bajo,Igyro=bajo
  t=14: STANDING                  | obs=Iacc=medio,Igyro=medio

log P(q*, O | λ) = -13.6533  →  P = 1.1761e-06


### E. Análisis e interpretación

**Sobre el vector π y la matriz A:**
Los parámetros estimados a partir de los datos reflejan que el estado STANDING concentra el 100% de la probabilidad inicial, mientras que los demás estados tienen una probabilidad nula. Esto indica que todas las secuencias de observación en el conjunto de datos comienzan con el sujeto en posición de pie (STANDING. Este comportamiento es coherente con la metodología de recolección de datos del experimento original, donde las capturas se estandarizaron para iniciar desde una postura estática de control.

**Sobre las observaciones:**
Durante la subsecuencia analizada de 15 instantes (donde el estado real constante es STANDING), las observaciones discretizadas se concentran consistentemente en niveles bajos de energía, predominando el índice Iacc=bajo, Igyro=bajo. Físicamente, esto refleja con precisión la naturaleza estática del estado: al estar de pie y en reposo, el sujeto no experimenta desplazamientos bruscos ni rotaciones corporales significativas. Esto valida el criterio de discretización elegido para separar actividades dinámicas de estáticas. Siendo congruentes con la matriz de transicion 

**Sobre Forward-Backward:**

Las probabilidades marginales obtenidas revelan una limitación al utilizar únicamente la magnitud de la aceleración y el giroscopio. Si bien el modelo descarta correctamente estados dinámicos (asignando 0.0% a WALKING en $t=0$), muestra confusión para distinguir entre posturas estáticas. Por ejemplo, en $t=3$ asigna un 26.3% de probabilidad a SITTING, y en $t=12$ un 8.8% a LAYING. Esto se explica porque las señales inerciales (ausencia de movimiento) son casi idénticas independientemente de si el sujeto está de pie, sentado o acostado.

**Sobre Viterbi:**
El algoritmo predice STANDING en los 15 instantes de tiempo, coincidiendo 
con el estado real en todos los casos. Esto es consecuencia directa de la 
combinación de π_STANDING dominante y A[STANDING,STANDING] ≈ 0.90: el 
camino de mayor probabilidad conjunta P(q*, O|λ) nunca abandona STANDING 
porque ninguna transición alternativa compensa la alta probabilidad de seguir en 
ese estado. La observación en t=14 (`Iacc=medio,Igyro=medio`, obs=4) es 
la única que podría sugerir un estado más dinámico, pero la inercia del 
modelo es suficiente para mantener STANDING como estado óptimo.

Esto demuestra la eficacia de Viterbi al buscar el camino óptimo global basándose en la matriz de transición. El modelo penaliza fuertemente las transiciones poco probables (como pasar de estar de pie a sentado y volver a pararse en fracciones de segundo), "suavizando" así la ambigüedad de las observaciones aisladas y priorizando la continuidad lógica del estado


